In [2]:
from pathlib import Path
import pandas as pd
from collections import defaultdict

# Specify the input data directory
data_dir = Path(".").absolute() / "eval-all" / "data-nesy-xlstm-raw"
data_dir

PosixPath('/workspaces/ts-qa/eval-all/data-nesy-xlstm-raw')

In [13]:
all_data = defaultdict(pd.DataFrame)

for folder in data_dir.iterdir():
    if not folder.is_dir():
        print(f"Skipping {folder}, not a directory")
        continue

    model_name = folder.name.split("_", 2)[2]
    replace_map = {
        "llama_xlstm": "NeSy xLSTM + Llama 3.1 (8B)",
        "qwen3_xlstm": "NeSy xLSTM + Qwen3 (8B)",
    }
    model_name = replace_map.get(model_name, model_name)
    print(f"Processing {folder.name}: {model_name}")

    # We have the model with and without ground truth (GT) actions, and the action encoder (AE)
    dfs_with_gt = [all_data[model_name + " GT"]]
    dfs_with_ae = [all_data[model_name + " AE"]]

    for file in folder.glob("*.jsonl"):
        df = pd.read_json(file, lines=True)
        if "_gt_" in file.name:
            dfs_with_gt.append(df.rename(columns={"pred_answer": "prediction_text"}))
        elif "_nogt_" in file.name:
            dfs_with_ae.append(df.rename(columns={"pred_answer": "prediction_text"}))
        else:
            print(f"Skipping {file.name}, not a GT or AE file")
            continue

    all_data[model_name + " GT"] = pd.concat(dfs_with_gt, ignore_index=True)
    all_data[model_name + " AE"] = pd.concat(dfs_with_ae, ignore_index=True)

print()
print(f"Total models processed: {len(all_data)}:\n{', '.join(list(all_data.keys()))}")

Skipping /workspaces/ts-qa/eval-all/data-nesy-xlstm-raw/combine.ipynb, not a directory
Processing binary_results_llama_xlstm: NeSy xLSTM + Llama 3.1 (8B)
Processing open_results_qwen3_xlstm: NeSy xLSTM + Qwen3 (8B)
Processing multi_results_llama_xlstm: NeSy xLSTM + Llama 3.1 (8B)
Processing multi_results_qwen3_xlstm: NeSy xLSTM + Qwen3 (8B)
Processing binary_results_qwen3_xlstm: NeSy xLSTM + Qwen3 (8B)
Processing open_results_llama_xlstm: NeSy xLSTM + Llama 3.1 (8B)

Total models processed: 4:
NeSy xLSTM + Llama 3.1 (8B) GT, NeSy xLSTM + Llama 3.1 (8B) AE, NeSy xLSTM + Qwen3 (8B) GT, NeSy xLSTM + Qwen3 (8B) AE


In [14]:
# Save as a CSV file, one per model
for model_name, df in all_data.items():
    df.to_csv(
        data_dir.parent / "data" / "4.4" / f"{model_name}_full_test.csv", index=False
    )
